In [4]:
#!pip install langchain langchain-core langchain-community

In [6]:
#!pip install pypdf pymupdf

In [5]:
#%pip install sentence-transformers chromadb

In [15]:
#%pip install langchain_text_splitters

In [6]:
from langchain_core.documents import Document

In [7]:
sample_doc = Document(
    page_content="hii Joti",
    metadata={"address": "Nawabgonj Dhaka"}
)

In [8]:
sample_doc

Document(metadata={'address': 'Nawabgonj Dhaka'}, page_content='hii Joti')

In [10]:
# Text data
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/file.txt", encoding="utf-8")

In [11]:
document = loader.load()

In [12]:
document

[Document(metadata={'source': 'data/file.txt'}, page_content='Regularization is a technique used in machine learning to prevent overfitting by adding a penalty to the model\'s complexity during training. It discourages the model from learning noise or minor details in the training data, allowing it to generalize better to new, unseen data. \nGeeksforGeeks\nGeeksforGeeks\n +2\nWhy It Matters\nWhen a model fits the training data too perfectly, it captures random fluctuations rather than the underlying pattern (high variance). Regularization introduces a bias-variance tradeoff, accepting a slight increase in training error to achieve a significant reduction in testing error. \nIBM\nIBM\n +4\nRegularization in Machine Learning | by Rishabh Singh | Medium\n17. Regularization: Controlling Overfitting in Machine ...\nRegularization in Machine Learning | Analytics Vidhya\nKey Regularization Techniques\nMost methods work by adding a penalty term to the loss function, controlled by a hyperparame

In [8]:
# PDF
from langchain_community.document_loaders import PyPDFLoader

#pdf_loader = PyPDFLoader("data/Rechearch_Paper.pdf")

#document = pdf_loader.load()

#document

In [9]:
import os

In [12]:
def load_all_pdfs() :
    folder_path = "data"
    num_doc = 0
    all_docs = []

    for filename in os.listdir(folder_path) :
        if filename.lower().endswith(".pdf") :
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_doc += 1

    print("total pdfs: ", num_doc)
    print("total pages: ", len(all_docs))
    return all_docs

In [13]:
all_pdf_documents = load_all_pdfs()

total pdfs:  1
total pages:  23


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents, chunk_size=500, chunk_overlap=50) :
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [17]:
chunks = split_doc(all_pdf_documents)

In [18]:
len(chunks)

187

In [ ]:
chunks

### Embedding

In [20]:
from sentence_transformers import SentenceTransformer

In [23]:
class EmbeddingManager :
    def __init__(self, model_name = "all-MiniLM-L6-v2") :
        self.model_name = model_name
        print("loading model.......", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text) :
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape: ", embeddings.shape)
        return embeddings

In [22]:
embedding_manager = EmbeddingManager()

loading model....... all-MiniLM-L6-v2


d:\Develops\Apna_College\Data-Science-interview\apna_college\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jotir\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7676.82it/s]


embedding dimensions =  384


C:\Users\jotir\AppData\Local\Temp\ipykernel_11000\4047445491.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions = ", self.model.get_sentence_embedding_dimension())


### Vector Store

In [24]:
import chromadb
import uuid

In [26]:
class VectorStoreManager :
    def __init__(self, persist_directory = "data/vector_store", collection_name = "pdf_documents") :
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_score()

    def _initialize_score(self) :
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description:" "vector store collection for pdf embeddings in RAG"}
        )
        print("initialized the vector with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings) :
        if len(documents) != len(embeddings) :
            raise ValueError("num of documents does not match num of embedings")
        
        # store -> ids, embeddings, documents, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)) :
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_lenght"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embedding_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embedding_list
            )

        print("total documents added in vector store = ", len(documents_content))
        print("docs in collection: ", self.collection.count())

In [27]:
vector_store = VectorStoreManager()

TypeError: argument 'metadata': 'set' object cannot be converted to 'PyDict'

In [ ]:
# data => documents => chunks => embeddings = > store in vector store

texts = [doc.page_content for doc in chunks]
embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

## Retrieval Pipeline

In [1]:
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
class RAGRetriever :
    def __init__(self, embedding_manager, vector_store) :
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retriever(self, query, top_k = 5, score_threshold = 0.0) :
        # query => emebedding
        query_embeddings = self.embedding_manager.generate_embeddings([query][0])

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine similariry
        retrieved_docs = []
        if results["document"] and results["documents"][0] :
            ids = results["ids"][0]
            metadatas = results["metadats"][0]
            documents = results["documents"][0]
            distance = result["distance"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distance)) :
                similariry = 1 - distance

                if similariry_score >= score_threshold :
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similariry_score,
                        "rank": i + 1
                    })
            print(f"retriever {len(retrieved_docs)} documents")
        else :
            print("no doucments found")

        return retrieved_docs

## Integrate with LLMs

In [1]:
API_KEY = "towhanvaloewgowe"

In [2]:
# !pip install langchain-openai

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    API_KEY,
    model = "gpt-5.4",
    temperature = 0.1,
    max_tokens = 1024
)

In [ ]:
# generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k = 3) :
    results = rag_retriever.retrieve(query, top_k)

    context= "\n".join(doc["document" for doc in results]) if results else ""
    if not context :
        print("we found no relevent context for the given query")
    
    # context + query
    prompt = f""" use given context to generate the answer
    context: {context}
    query: {query} """

    response = llm.invoke(prompt)
    return response.document

SyntaxError: incomplete input (2407180396.py, line 2)

In [ ]:
answer = generate_output("what is encoder-decoder", rag_retriever, llm)